Get averages for times of day

In [4]:
#read in relevant packages
import xarray as xr
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import pyproj #for utm
#to make sure you can see the env: https://stackoverflow.com/questions/53004311/how-to-add-conda-environment-to-jupyter-lab 

from netCDF4 import Dataset #for using wrf.getvar

##dates
import datetime 

#to get a list of the files in the folder
import glob #glob2

##attempt to reformat the netcdf in one line
import xwrf #https://xwrf.readthedocs.io/en/latest/how-to/install-xwrf.html

#to quickly calculate relative humidity
from wrf import getvar, interplevel, to_np, latlon_coords, uvmet
import wrf #https://wrf-python.readthedocs.io/en/latest/


## Overall Plan

* get the variables from the wrfout files
* get the hourly averages
* output it into an array format that is a bit easier to use for analyses/some visuals

Relies heavily on wrf-python package. So far we are focusing on elements of the atmospheric conditions that could relate to MHW impacts (or have been seen to do so in other contexts)
* temperature
* relative humidity
* wind speed and direction
* pressure levels (including sea level pressure)
* low level cloud cover

In [5]:
##trying this with wrf-python instead to see if it makes it at all easier
d02_2016_names = glob.glob("../../01Data/Houston_MarineHeatWave_2016/wrfout_MarineHeatWave.d02.*")
#d02_2017_names = glob.glob("../../01Data/Houston_MarineHeatWave_2017/wrfout_MarineHeatWave.d02.*")
#d02_2017_climavg_names  = glob.glob("../../01Data/Houston_MarineHeatWave_2017_w16SST/wrfout_MarineHeatWave.d02.*")

##Ideally we want to ignore the first few days as spin up (maybe get rid of 2 days worth)

#pull out the datasets
wrfin = [Dataset(f) for f in d02_2016_names[8:] ] #gives you two days of burn-in time I think
initial_ds = xr.open_dataset(d02_2016_names[0]).xwrf.postprocess() #for land use
#wrfin = [Dataset(f) for f in d02_2017_names[8:] ] #gives you two days of burn-in time I think
#initial_ds = xr.open_dataset(d02_2017_names[0]).xwrf.postprocess() #for land use
#wrfin = [Dataset(f) for f in d02_2017_climavg_names[8:] ] #gives you two days of burn-in time I think
#initial_ds = xr.open_dataset(d02_2017_climavg_names[0]).xwrf.postprocess() #for land use

T2_data = wrf.getvar(wrfin, "T2", timeidx=wrf.ALL_TIMES)
rh2_data = wrf.getvar(wrfin, "rh2", timeidx=wrf.ALL_TIMES)

#Uwind_data = wrf.getvar(wrfin, "U", timeidx=wrf.ALL_TIMES)
#Vwind_data = wrf.getvar(wrfin, "V", timeidx=wrf.ALL_TIMES)

# Get grid-relative U and V
uv = getvar(wrfin, "uvmet", timeidx=wrf.ALL_TIMES) 
uv10m = getvar(wrfin, "uvmet10", timeidx=wrf.ALL_TIMES)

#wind speed
wspd =  getvar(wrfin, "uvmet_wspd_wdir", units="m s-1", timeidx=wrf.ALL_TIMES)[0,:]
wspd10 =  getvar(wrfin, "uvmet_wspd_wdir", units="m s-1", timeidx=wrf.ALL_TIMES)[0,:]

#pressure
p = getvar(wrfin, "pressure", timeidx=wrf.ALL_TIMES)         # shape: (Time, bottom_top, south_north, west_east)
uv_850 = interplevel(uv, p, 850)

#trying to get clouds
#these are the thresholds with height, but we don't have that, we have pressure: https://www.weather.gov/key/cloudchart
clouds = getvar(wrfin, "cloudfrac", timeidx=wrf.ALL_TIMES)  
llclouds = clouds[0,:,:,:]

#sea level pressure
slp = wrf.getvar(wrfin, "slp", timeidx=wrf.ALL_TIMES)
#have to manage the shortwave accumulation a bit differently here...

IndexError: list index out of range

Now go and extract the values of these variables across the whole grid for each hour

In [3]:
##now get the average values for each hour
DatesList = T2_data.coords['Time'].values
HourList = pd.to_datetime(DatesList).hour

#prep data for land use
init_ds = initial_ds.isel(Time=0)
landuse = init_ds.LU_INDEX

# Variables and datasets
variables = {
    'T2': T2_data,
    'rh2': rh2_data,
    'uv10': uv10m,
    'uv850': uv_850,
    'llc': llclouds,
    'wspd': wspd,
    'wspd10': wspd10,
    'slp': slp,
}
# Prepare storage for results
results = []

# Loop over hours
for h in range(24):
    
    hr_pos = np.where(HourList == h)[0]
    
    if len(hr_pos) == 0:
        continue

    var_means = {}

    # Compute mean across time for each variable
    for var_name, data in variables.items():
        selected = data.isel(Time=hr_pos)
        
        if var_name == 'sw' and h == 0:
            hr_pos = hr_pos[1:]  # skip first time point for SW
            selected = data.sel(Time=hr_pos)
            
        var_means[var_name] = selected.mean(dim='Time', skipna=True)

    # Grid info
    lat_vals = var_means['T2'].coords['XLAT'].values
    lon_vals = var_means['T2'].coords['XLONG'].values

    ny, nx = landuse.shape  # typically (south_north, west_east)
    uv10 = var_means["uv10"]
    uv850 = var_means["uv850"]
    wspd = var_means["wspd"]
    wspd10 = var_means["wspd10"]

    for i in range(ny):       # south_north
        for j in range(nx):   # west_east
            lat = lat_vals[i, j]
            lon = lon_vals[i, j]

            #for the winds
            row = {
                'hour': h,
                'lat': float(lat),
                'lon': float(lon),
                'landuse': int(landuse.values[i, j])
            }
            
            row["u10"] = uv10.values[0, i, j].item()
            row["v10"] = uv10.values[1, i, j].item()
            row["u850"] = uv850.values[0, i, j].item()
            row["v850"] = uv850.values[1, i, j].item()
            row["wspd0"] = wspd.values[0, i, j].item()
            row["wspd10"] = wspd10.values[0, i, j].item()
            
            for var_name, mean_arr in var_means.items():

                if var_name =="uv10" or var_name =="uv850" or var_name =="wspd" or var_name =="wspd10":
                    continue
                    
                row[var_name] = mean_arr.values[i, j].item()
            results.append(row)


# Convert to DataFrame
df = pd.DataFrame(results)


# Optional: save to CSV
#df.to_csv("hourly_grid_averages.csv", index=False)

In [4]:
df.to_csv('../../03ProcessedData/LargeFiles/HourlyMeans_2016_070125.csv', index=False)
#df.to_csv('../../03ProcessedData/LargeFiles/HourlyMeans_2017_070125.csv', index=False)
#df.to_csv('../../03ProcessedData/LargeFiles/HourlyMeans_2017_w16SST_070125.csv', index=False)